In [1]:
import math, numpy as np, pandas as pd
data = pd.DataFrame({
 'Wind':['Weak','Strong','Weak','Weak','Strong'],
 'Humidity':['High','High','Low','Low','Low'],
 'Rain':['Yes','Yes','No','Yes','No']})
print(data)

     Wind Humidity Rain
0    Weak     High  Yes
1  Strong     High  Yes
2    Weak      Low   No
3    Weak      Low  Yes
4  Strong      Low   No


In [2]:
def nb_predict(df, query, alpha=1):
    n=len(df); res={}
    for c in df['Rain'].unique():
        sub=df[df['Rain']==c]; p=len(sub)/n
        for f,v in query.items():
            k=df[f].nunique()
            like=((sub[f]==v).sum()+alpha)/(len(sub)+alpha*k)
            print(f'P({f}={v}|Rain={c}) = ({(sub[f]==v).sum()}+{alpha})/({len(sub)}+{alpha*k}) = {like:.4f}')
            p*=like
        res[c]=p
        print(f'  unnormalised score for Rain={c}: {p:.4f}\n')
    tot=sum(res.values())
    return {c:round(v/tot,4) for c,v in res.items()}
post=nb_predict(data,{'Wind':'Strong','Humidity':'Low'})
print(post)

P(Wind=Strong|Rain=Yes) = (1+1)/(3+2) = 0.4000
P(Humidity=Low|Rain=Yes) = (1+1)/(3+2) = 0.4000
  unnormalised score for Rain=Yes: 0.0960

P(Wind=Strong|Rain=No) = (1+1)/(2+2) = 0.5000
P(Humidity=Low|Rain=No) = (2+1)/(2+2) = 0.7500
  unnormalised score for Rain=No: 0.1500

{'Yes': np.float64(0.3902), 'No': np.float64(0.6098)}


In [3]:
def entropy(s):
    p=s.value_counts(normalize=True)
    return -sum(x*math.log2(x) for x in p if x>0)
def gain(df,f):
    H=entropy(df['Rain']); w=0
    for v,sub in df.groupby(f):
        h=entropy(sub['Rain']); w+=len(sub)/len(df)*h
        print(f'  {f}={v}: n={len(sub)}, entropy={h:.4f}')
    return H-w
print('Entropy of Rain =',round(entropy(data['Rain']),4))
for f in ['Wind','Humidity']:
    print(f,'IG =',round(gain(data,f),4))
low=data[data['Humidity']=='Low']
print('Inside Humidity=Low, IG(Wind) =',round(gain(low,'Wind'),4))

Entropy of Rain = 0.971
  Wind=Strong: n=2, entropy=1.0000
  Wind=Weak: n=3, entropy=0.9183
Wind IG = 0.02
  Humidity=High: n=2, entropy=-0.0000
  Humidity=Low: n=3, entropy=0.9183
Humidity IG = 0.42
  Wind=Strong: n=1, entropy=-0.0000
  Wind=Weak: n=2, entropy=1.0000
Inside Humidity=Low, IG(Wind) = 0.2516


In [4]:
X=np.array([[0,1],[1,1],[0,0],[0,0],[1,0]],float); y=np.array([1,1,0,1,0],float)
sig=lambda z:1/(1+np.exp(-z))
w=np.zeros(2); b=0.0

p=sig(X@w+b); print('p at start:',p)
print('grad_w =',X.T@(p-y)/5,' grad_b =',(p-y).mean())
w1=w-0.1*X.T@(p-y)/5; b1=b-0.1*(p-y).mean(); print('after 1 step: w =',w1,'b =',b1)

lr=0.5
for i in range(5000):
    p=sig(X@w+b); w-=lr*X.T@(p-y)/5; b-=lr*(p-y).mean()
print('w =',w.round(3),'b =',round(b,3))
q=np.array([1,0]); pr=sig(w@q+b); print('P(rain | Strong, Low) =',round(pr,4))

p at start: [0.5 0.5 0.5 0.5 0.5]
grad_w = [ 0.  -0.2]  grad_b = -0.1
after 1 step: w = [0.   0.02] b = 0.010000000000000002
w = [-5.113 10.646] b = -0.004
P(rain | Strong, Low) = 0.006


In [5]:
comp=pd.DataFrame({
 'Algorithm':['Naive Bayes (Laplace)','Decision tree (IG)','Logistic regression'],
 'P(rain | Strong, Low)':[round(post['Yes'],3),'0 (leaf = No)',round(pr,3)],
 'Decision':['No rain -> deliver']*3,
 'Output type':['probability','hard class','probability'],
 'Handles unseen combos':['yes (smoothing)','only via learned splits','yes (weights)']})
print(comp.to_string())

               Algorithm P(rain | Strong, Low)            Decision  Output type    Handles unseen combos
0  Naive Bayes (Laplace)                  0.39  No rain -> deliver  probability          yes (smoothing)
1     Decision tree (IG)         0 (leaf = No)  No rain -> deliver   hard class  only via learned splits
2    Logistic regression                 0.006  No rain -> deliver  probability            yes (weights)
